## Scratch version vs. LangGraph RAG version

### Architecture

| Concern | Scratch version | LangGraph RAG version |
|---|---|---|
| **Loop control** | Manual `while not done and step < max_steps` | LangGraph `StateGraph` — nodes + conditional edges declare the flow |
| **State** | Local Python variables (`memory`, `turn_history`, `step`, `done`) | `AgentState` TypedDict — fully serialisable, passed between nodes |
| **Routing logic** | `if/continue/break` inside the loop | `route_after_reflect` conditional edge — routing is explicit and inspectable |
| **Final answer** | `decision.reasoning` (whatever the LLM put in the JSON field) | Dedicated `generate_answer` node — a separate focused LLM call with full memory + a final retrieval pass |
| **Checkpointing** | None | Optional `MemorySaver` — state persisted between invocations, resumable by `thread_id` |
| **Observability** | Logger inside the loop | LangGraph streams node-completion events; graph topology exportable as Mermaid diagram |

---

### Knowledge grounding

| | Scratch version | LangGraph RAG version |
|---|---|---|
| **Knowledge source** | Only what the model was trained on + tool results added to memory | RAG vector index built before the loop (my_company doc + prospect profile + web-search docs) |
| **Retrieval** | None — full task + memory pasted into every prompt | `top_k` chunks retrieved from vector store at every Reason and Reflect step |
| **Embeddings** | None | `all-MiniLM-L6-v2` via sentence-transformers (local, no API key); Ollama embeddings optional |
| **Vector store** | None | `InMemoryVectorStore` (default) or Chroma (persisted to disk) |
| **Live search → store** | Search result goes into memory string only | After every `search_web` call, results are also embedded and added to the vector store so all future retrievals see them |
| **Initial search** | No | Optional one-shot search on `"{company} {industry}"` before the loop to pre-seed the index |

---

### Tools

| Tool | Scratch version | LangGraph RAG version |
|---|---|---|
| `search_web` | ✅ DuckDuckGo, result added to memory string | ✅ Same, **plus** results embedded into vector store |
| `extract_insights` | ✅ LLM over profile text | ✅ Same |
| `save_notes` | ✅ Explicitly saves findings into persistent notes / `AgentMemory` | ❌ Not present — replaced by the `update` node, which appends every step's observation to `memory_items` automatically. The vector store also serves as persistent knowledge. |

The `save_notes` tool in the scratch version was a deliberate act — the model had to *choose* to save something. In the LangGraph version, saving is structural: `update_node` runs unconditionally every cycle and the vector store accumulates search docs automatically. The trade-off is that scratch's `save_notes` let the model be selective about what was worth remembering, while the current version saves everything.

---

### Code organisation

| | Scratch version | LangGraph RAG version |
|---|---|---|
| `agent_loop.py` | ~580 lines: loop logic, step functions, logging helpers, entry point | ~160 lines: RAG setup, logging scaffold, `run_sales_rep_flow` only |
| `graph.py` | ❌ | ✅ NEW — all node functions, `AgentState`, routing, `build_graph()` |
| `rag.py` | ❌ | ✅ NEW — document building, chunking, vector store, retriever |
| `decisions.py` | ✅ | ✅ Same |
| `memory.py` | ✅ `AgentMemory` used directly in the loop | ✅ Present but `memory_items: List[str]` in state replaces it at runtime |

---